# Telecom Network Operations — Pre-Filled Configuration
### Ready-to-Run Config for the Telecom Use Case

This is a **pre-configured** version of `00_Industry_Config` specifically for the
**Telecom Network Operations Documentation Burden** demo. All table lists, artifact names,
and data paths are hardcoded from the existing 23-CSV telecom dataset.

**Use this instead of `00_Industry_Config` if you want to skip auto-discovery and run
directly with the known telecom tables.**

---

### Data Summary
- **6 Dimensions** — Engineers, Service Areas, Network Segments, Ticket Types, Enterprise Customers, Equipment
- **6 Batch Facts** — Trouble Tickets, Field Dispatches, Technician Wellness, Ticket Quality, Customer Satisfaction, Network Performance
- **6 Event Facts** → Eventhouse — NMS Interactions, Outage Events, RCA Documents, Ticket Escalations, SLA Alerts, Field Location
- **5 Streams** → Eventhouse — Network Alarms, Ticket Activity, Field Dispatch, Performance Metrics, Customer Impact

### Ontology
- **6 Entity Types:** Engineer, ServiceArea, NetworkSegment, TicketType, EnterpriseCustomer, Equipment
- **5 Relationship Types:** assigned_to, serves_area, monitors_segment, supports_customer, operates_equipment
- **6 Contextualizations:** TroubleTicket, OutageEvent, RCADocument, FieldDispatch, SLAAlert, NMSInteraction

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# INDUSTRY SETTING
# ════════════════════════════════════════════════════════════════════════

INDUSTRY       = "telecom"
INDUSTRY_LABEL = "Telecom"

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# FABRIC ARTIFACT NAMES
# ════════════════════════════════════════════════════════════════════════
# Update these to match your Fabric workspace artifact names.

LAKEHOUSE_NAME      = "TelecomLakehouse"
WAREHOUSE_NAME      = "Telecom_Datawarehouse"
EVENTHOUSE_NAME     = "telecom_rt_store"
ONTOLOGY_NAME       = "TelecomNetOpsOntology"
DATA_AGENT_NAME     = "TelecomQA"
OPS_AGENT_NAME      = "TelecomDocBurden"
SEMANTIC_MODEL_NAME = "Telecom_Network_Ops_Model"

print(f"Lakehouse:      {LAKEHOUSE_NAME}")
print(f"Warehouse:      {WAREHOUSE_NAME}")
print(f"Eventhouse:     {EVENTHOUSE_NAME}")
print(f"Ontology:       {ONTOLOGY_NAME}")
print(f"Data Agent:     {DATA_AGENT_NAME}")
print(f"Ops Agent:      {OPS_AGENT_NAME}")
print(f"Semantic Model: {SEMANTIC_MODEL_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# DATA PATHS & EVENTHOUSE CONNECTION
# ════════════════════════════════════════════════════════════════════════

# CSV files location in Lakehouse Files area
CSV_BASE_PATH = "/lakehouse/default/Files/telecom_network_operations/data"

# Schemas
LAKEHOUSE_SCHEMA = "dbo"
WAREHOUSE_SCHEMA = "dbo"

# ── Eventhouse Connection ───────────────────────────────────────────────
# Fill these in from your Fabric workspace
EVENTHOUSE_CLUSTER_URI = ""   # e.g. "https://trd-xxxxx.z5.kusto.fabric.microsoft.com"
EVENTHOUSE_DATABASE    = ""   # e.g. "telecom_rt_store"

# Ontology package path
ONTOLOGY_PACKAGE_PATH = "/lakehouse/default/Files/telecom_network_ops_ontology_package.iq"

print(f"CSV source:     {CSV_BASE_PATH}")
print(f"LH schema:      {LAKEHOUSE_SCHEMA}")
print(f"WH schema:      {WAREHOUSE_SCHEMA}")
print(f"Eventhouse:     {EVENTHOUSE_CLUSTER_URI or '(fill in your cluster URI)'}")
print(f"Ontology pkg:   {ONTOLOGY_PACKAGE_PATH}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# TELECOM TABLE DEFINITIONS (pre-filled, no auto-discovery needed)
# ════════════════════════════════════════════════════════════════════════

import os, json

# ── 6 Dimension Tables → Lakehouse + Warehouse ──────────────────────────
DIM_TABLES = [
    "dim_engineers",
    "dim_service_areas",
    "dim_network_segments",
    "dim_ticket_types",
    "dim_enterprise_customers",
    "dim_equipment",
]

# ── 6 Batch Fact Tables → Lakehouse + Warehouse ─────────────────────────
FACT_TABLES_BATCH = [
    "fact_trouble_tickets",
    "fact_field_dispatches",
    "fact_technician_wellness",
    "fact_ticket_quality",
    "fact_customer_satisfaction",
    "fact_network_performance",
]

# ── 6 Event Fact Tables → Eventhouse (KQL) ──────────────────────────────
FACT_TABLES_EVENT = [
    "fact_nms_interactions",
    "fact_outage_events",
    "fact_rca_documents",
    "fact_ticket_escalations",
    "fact_sla_alerts",
    "fact_field_location",
]

# ── 5 Streaming Tables → Eventhouse (KQL) ───────────────────────────────
STREAM_TABLES = [
    "stream_network_alarms",
    "stream_ticket_activity",
    "stream_field_dispatch",
    "stream_performance_metrics",
    "stream_customer_impact",
]

# ── Combined Target Lists ───────────────────────────────────────────────
LAKEHOUSE_TABLES  = DIM_TABLES + FACT_TABLES_BATCH
WAREHOUSE_TABLES  = DIM_TABLES + FACT_TABLES_BATCH + FACT_TABLES_EVENT
EVENTHOUSE_TABLES = FACT_TABLES_EVENT + STREAM_TABLES

# ── KQL Table Name Mapping (CSV name → KQL table name) ─────────────────
# Event facts strip the 'fact_' prefix; streams strip 'stream_' prefix
EVENTHOUSE_KQL_NAMES = {
    "fact_nms_interactions":       "nms_interactions",
    "fact_outage_events":          "outage_events",
    "fact_rca_documents":          "rca_documents",
    "fact_ticket_escalations":     "ticket_escalations",
    "fact_sla_alerts":             "sla_alerts",
    "fact_field_location":         "field_location",
    "stream_network_alarms":       "network_alarms",
    "stream_ticket_activity":      "ticket_activity",
    "stream_field_dispatch":       "field_dispatch",
    "stream_performance_metrics":  "performance_metrics",
    "stream_customer_impact":      "customer_impact",
}

print(f"{'='*60}")
print(f"TELECOM TABLE INVENTORY")
print(f"{'='*60}")
print(f"\nDimension tables ({len(DIM_TABLES)}):")
for t in DIM_TABLES: print(f"  • {t}")
print(f"\nFact tables — Batch ({len(FACT_TABLES_BATCH)}):")
for t in FACT_TABLES_BATCH: print(f"  • {t}")
print(f"\nFact tables — Event/Eventhouse ({len(FACT_TABLES_EVENT)}):")
for t in FACT_TABLES_EVENT: print(f"  • {t}  →  {EVENTHOUSE_KQL_NAMES[t]}")
print(f"\nStreaming tables — Eventhouse ({len(STREAM_TABLES)}):")
for t in STREAM_TABLES: print(f"  • {t}  →  {EVENTHOUSE_KQL_NAMES[t]}")
print(f"\n{'─'*60}")
print(f"Lakehouse target:  {len(LAKEHOUSE_TABLES)} tables (12 Delta tables)")
print(f"Warehouse target:  {len(WAREHOUSE_TABLES)} tables (18 SQL tables)")
print(f"Eventhouse target: {len(EVENTHOUSE_TABLES)} tables (11 KQL tables)")
print(f"Total:             23 CSV files")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# EXPECTED ROW COUNTS (for validation in downstream notebooks)
# ════════════════════════════════════════════════════════════════════════

EXPECTED_ROW_COUNTS = {
    # Dimensions
    "dim_engineers":              30,
    "dim_service_areas":          15,
    "dim_network_segments":       12,
    "dim_ticket_types":           18,
    "dim_enterprise_customers":   25,
    "dim_equipment":              50,
    # Batch Facts
    "fact_trouble_tickets":       300,
    "fact_field_dispatches":      150,
    "fact_technician_wellness":    30,
    "fact_ticket_quality":        300,
    "fact_customer_satisfaction":  25,
    "fact_network_performance":    60,
    # Event Facts
    "fact_nms_interactions":      500,
    "fact_outage_events":          40,
    "fact_rca_documents":          40,
    "fact_ticket_escalations":     80,
    "fact_sla_alerts":            100,
    "fact_field_location":        500,
    # Streams
    "stream_network_alarms":      100,
    "stream_ticket_activity":     100,
    "stream_field_dispatch":       70,
    "stream_performance_metrics": 100,
    "stream_customer_impact":      60,
}

total_lakehouse = sum(EXPECTED_ROW_COUNTS[t] for t in LAKEHOUSE_TABLES)
total_eventhouse = sum(EXPECTED_ROW_COUNTS[t] for t in EVENTHOUSE_TABLES)
print(f"Expected Lakehouse rows:  {total_lakehouse:,}")
print(f"Expected Eventhouse rows: {total_eventhouse:,}")
print(f"Expected total rows:      {sum(EXPECTED_ROW_COUNTS.values()):,}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SCHEMA INSPECTION HELPER
# ════════════════════════════════════════════════════════════════════════

def preview_table(table_name, base_path=CSV_BASE_PATH, rows=5):
    """Read a CSV and display schema + sample rows."""
    path = f"{base_path}/{table_name}.csv"
    df = spark.read.option("header", True).option("inferSchema", True).csv(path)
    print(f"\n{'─'*60}")
    print(f"Table: {table_name}  |  {df.count()} rows  |  {len(df.columns)} columns")
    print(f"{'─'*60}")
    df.printSchema()
    df.show(rows, truncate=False)
    return df

# Uncomment to preview specific tables:
# preview_table("dim_engineers")
# preview_table("fact_trouble_tickets")
# preview_table("stream_network_alarms")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CONFIG SUMMARY (exported to downstream notebooks via %run)
# ════════════════════════════════════════════════════════════════════════

CONFIG = {
    "industry":             INDUSTRY,
    "industry_label":       INDUSTRY_LABEL,
    "lakehouse_name":       LAKEHOUSE_NAME,
    "warehouse_name":       WAREHOUSE_NAME,
    "eventhouse_name":      EVENTHOUSE_NAME,
    "eventhouse_uri":       EVENTHOUSE_CLUSTER_URI,
    "eventhouse_db":        EVENTHOUSE_DATABASE,
    "ontology_name":        ONTOLOGY_NAME,
    "ontology_package":     ONTOLOGY_PACKAGE_PATH,
    "data_agent_name":      DATA_AGENT_NAME,
    "ops_agent_name":       OPS_AGENT_NAME,
    "semantic_model_name":  SEMANTIC_MODEL_NAME,
    "csv_base_path":        CSV_BASE_PATH,
    "lakehouse_schema":     LAKEHOUSE_SCHEMA,
    "warehouse_schema":     WAREHOUSE_SCHEMA,
    "dim_tables":           DIM_TABLES,
    "fact_tables_batch":    FACT_TABLES_BATCH,
    "fact_tables_event":    FACT_TABLES_EVENT,
    "stream_tables":        STREAM_TABLES,
    "lakehouse_tables":     LAKEHOUSE_TABLES,
    "warehouse_tables":     WAREHOUSE_TABLES,
    "eventhouse_tables":    EVENTHOUSE_TABLES,
}

print(f"\n{'═'*60}")
print(f"✅ TELECOM CONFIG READY")
print(f"{'═'*60}")
print(json.dumps({k: v if not isinstance(v, list) else f"{len(v)} tables" for k, v in CONFIG.items()}, indent=2))
print(f"\nThis config is imported by downstream notebooks via:")
print(f"  %run ./Telecom_Config")